<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/12-supervised-learning/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 12: Solutions

Try the exercises first. The notes after each solution explain the maths behind it and the mistake each check is designed to catch.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import math
import warnings

import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, (pd.DataFrame, pd.Series)) or isinstance(b, (pd.DataFrame, pd.Series)):
        if type(a) is not type(b):
            return False
        options = dict(check_dtype=False, check_exact=False, rtol=tol, atol=tol, check_index_type=False)
        try:
            if isinstance(a, pd.DataFrame):
                pd.testing.assert_frame_equal(a, b, check_column_type=False, **options)
            else:
                pd.testing.assert_series_equal(a, b, **options)
            return True
        except AssertionError:
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        lines = (x.to_string(max_rows=6, max_cols=6) if isinstance(x, pd.DataFrame) else x.to_string(max_rows=6)).splitlines()
        return f"{type(x).__name__} shape={x.shape}\n          " + "\n          ".join(lines[:8])
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return text
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def check_regression_metrics(f):
    _scenarios("regression_metrics", [
        ("a simple example", lambda: f([3, 5, 7, 9], [4, 5, 6, 11]),
         {"mae": 1.0, "rmse": math.sqrt(1.5), "r2": 1 - 6 / 20}, "MAE = mean |e|, RMSE = sqrt(mean e²), R² = 1 - SSE/SST"),
        ("a perfect model", lambda: f([1.0, 2.0, 3.0], [1.0, 2.0, 3.0]), {"mae": 0.0, "rmse": 0.0, "r2": 1.0}, None),
        ("predicting the mean gives R² = 0", lambda: f([1, 2, 3, 6], [3, 3, 3, 3])["r2"], 0.0, None),
        ("worse than the mean gives a negative R²", lambda: f([1, 2, 3], [3, 4, 5])["r2"], -5.0,
         "nothing stops R² going below 0"),
        ("returns floats", lambda: [type(v) for v in f(np.array([1, 2]), np.array([1, 3])).values()], [float, float, float], None),
        ("different lengths raise", lambda: f([1, 2, 3], [1, 2]), Raises(ValueError), None),
        ("empty input raises", lambda: f([], []), Raises(ValueError), None),
        ("constant y_true raises", lambda: f([4, 4, 4], [4, 5, 3]), Raises(ValueError), "R² divides by the spread of y_true"),
    ])


def check_polynomial_features(f):
    _scenarios("polynomial_features", [
        ("the example", lambda: f([1, 2, 3], 3), np.array([[1, 1, 1], [2, 4, 8], [3, 9, 27]], dtype=float),
         "columns x, x², ..., x^degree"),
        ("degree 1 is a single column", lambda: f(np.array([0.5, -2.0]), 1), np.array([[0.5], [-2.0]]), None),
        ("shape (n, degree), no column of ones", lambda: f(np.linspace(0, 1, 7), 4).shape, (7, 4), "start the powers at 1, not 0"),
        ("negative inputs", lambda: f([-2], 3), np.array([[-2.0, 4.0, -8.0]]), None),
        ("returns floats", lambda: f([1, 2], 2).dtype.kind, "f", None),
        ("degree 0 raises", lambda: f([1, 2], 0), Raises(ValueError), None),
    ])


def check_fit_least_squares(f):
    rng = np.random.default_rng(0)
    X = rng.normal(size=(50, 3))
    y = X @ np.array([2.0, -1.0, 0.5]) + 4 + rng.normal(0, 0.1, 50)
    ref = LinearRegression().fit(X, y)
    X_dup = np.column_stack([X[:, 0], X[:, 0], X[:, 1]])

    def duplicate_predictions():
        w, b = f(X_dup, y)
        return bool(np.allclose(X_dup @ w + b, LinearRegression().fit(X_dup, y).predict(X_dup), atol=1e-8))

    X_near = np.column_stack([X[:, 0], X[:, 0] + 1e-9 * rng.normal(size=50), X[:, 1]])

    def near_duplicate_residual():
        w, b = f(X_near, y)
        best = np.linalg.lstsq(np.column_stack([X_near, np.ones(50)]), y, rcond=None)[0]
        best_rmse = np.sqrt(np.mean((np.column_stack([X_near, np.ones(50)]) @ best - y) ** 2))
        return bool(np.sqrt(np.mean((X_near @ w + b - y) ** 2)) <= best_rmse + 1e-7)

    _scenarios("fit_least_squares", [
        ("matches scikit-learn", lambda: (lambda wb: (wb[0], wb[1]))(f(X, y)), (ref.coef_, ref.intercept_), None),
        ("an exact line is recovered", lambda: f([0, 1, 2, 3], [1, 3, 5, 7]), (np.array([2.0]), 1.0),
         "a 1D X is a single feature: reshape it to (n, 1)"),
        ("b is a float", lambda: type(f(X, y)[1]), float, "return float(coefficients[-1])"),
        ("a duplicated column still gives the right predictions", duplicate_predictions, True,
         "use np.linalg.lstsq; the normal equations fail when XᵀX is singular"),
        ("an almost-duplicated column: the fit is as good as the best possible", near_duplicate_residual, True,
         "the normal equations lose accuracy when columns are nearly identical; lstsq doesn't"),
    ])


def check_add_interactions(f):
    df = pd.DataFrame({"income": [1.0, 2.0, 3.0], "rooms": [4.0, 5.0, 6.0], "age": [10, 20, 30]})

    def unchanged():
        before = df.copy()
        f(df, [("income", "rooms")])
        return df.equals(before)

    _scenarios("add_interactions", [
        ("adds one product column", lambda: f(df, [("income", "rooms")])["income*rooms"].tolist(), [4.0, 10.0, 18.0], None),
        ("columns named 'a*b', in order, after the originals", lambda: list(f(df, [("income", "age"), ("rooms", "age")]).columns),
         ["income", "rooms", "age", "income*age", "rooms*age"], None),
        ("the input is not modified", unchanged, True, "work on df.copy()"),
        ("no pairs: same columns", lambda: list(f(df, []).columns), ["income", "rooms", "age"], None),
        ("a missing column raises KeyError", lambda: f(df, [("income", "size")]), Raises(KeyError), None),
    ])


def check_ridge_closed_form(f):
    rng = np.random.default_rng(1)
    X = rng.normal(size=(60, 4))
    y = X @ np.array([1.0, 2.0, -1.0, 0.0]) + 150 + rng.normal(0, 0.5, 60)

    def matches(alpha):
        w, b = f(X, y, alpha)
        ref = Ridge(alpha=alpha).fit(X, y)
        return bool(np.allclose(w, ref.coef_, atol=1e-8) and abs(b - ref.intercept_) < 1e-8)

    def huge():
        w, b = f(X, y, 1e10)
        return (bool(np.allclose(w, 0, atol=1e-6)), round(b, 4) == round(y.mean(), 4))

    def duplicates():
        Xd = np.column_stack([X[:, 0], X[:, 0], X[:, 1]])
        w, b = f(Xd, y, 1.0)
        ref = Ridge(alpha=1.0).fit(Xd, y)
        return bool(np.allclose(w, ref.coef_, atol=1e-8))

    ols = LinearRegression().fit(X, y)
    _scenarios("ridge_closed_form", [
        ("alpha = 0 is ordinary least squares", lambda: f(X, y, 0.0), (ols.coef_, ols.intercept_), None),
        ("matches scikit-learn's Ridge, alpha = 5", lambda: matches(5.0), True, "solve (Xcᵀ Xc + αI) w = Xcᵀ yc on centred data"),
        ("matches scikit-learn's Ridge, alpha = 100", lambda: matches(100.0), True, None),
        ("huge alpha: weights near 0, intercept stays at mean(y)", huge, (True, True),
         "don't penalise the intercept: centre, solve for w, then b = mean(y) - mean(X)·w"),
        ("handles a duplicated column", duplicates, True, None),
    ])


def check_choose_alpha(f):
    X, y = load_diabetes(return_X_y=True)
    cv = KFold(5, shuffle=True, random_state=0)
    alphas = [0.01, 1.0, 10.0, 100.0, 1000.0]
    ref = pd.Series({a: -cross_val_score(make_pipeline(StandardScaler(), Ridge(alpha=a)), X, y, cv=cv,
                                         scoring="neg_root_mean_squared_error").mean() for a in alphas})
    result = {}

    def run():
        if "v" not in result:
            result["v"] = f(X, y, alphas, cv)
        return result["v"]

    _scenarios("choose_alpha", [
        ("RMSE for each alpha matches a reference (scaler inside the pipeline)", lambda: run()[1].round(6).tolist(),
         ref.round(6).tolist(), "make_pipeline(StandardScaler(), Ridge(alpha=a)), and flip the sign of the scores"),
        ("indexed by alpha, in order", lambda: list(run()[1].index), alphas, None),
        ("best alpha is the one with the lowest RMSE", lambda: run()[0], ref.idxmin(), "rmse.idxmin()"),
    ])


def check_learning_curve_points(f):
    rng = np.random.default_rng(2)
    x = rng.uniform(0, 4, 300)
    X = x[:, None]
    y = np.sin(1.5 * x) + rng.normal(0, 0.3, 300)
    cv = KFold(5, shuffle=True, random_state=0)
    sizes = [10, 40, 160, 240]

    ref = {}
    for n in sizes:
        tr_e, va_e = [], []
        for tr, va in cv.split(X):
            sub = tr[:n]
            m = LinearRegression().fit(X[sub], y[sub])
            tr_e.append(np.sqrt(np.mean((m.predict(X[sub]) - y[sub]) ** 2)))
            va_e.append(np.sqrt(np.mean((m.predict(X[va]) - y[va]) ** 2)))
        ref[n] = (np.mean(tr_e), np.mean(va_e))

    out = {}

    def run():
        if "v" not in out:
            out["v"] = f(LinearRegression(), X, y, sizes, cv)
        return out["v"]

    def model_unfitted():
        model = LinearRegression()
        f(model, X, y, [20], cv)
        return hasattr(model, "coef_")

    _scenarios("learning_curve_points", [
        ("columns and index", lambda: (list(run().columns), list(run().index)), (["train_rmse", "val_rmse"], sizes), None),
        ("values match a reference", lambda: [tuple(np.round(run().loc[n].to_numpy(), 6)) for n in sizes],
         [tuple(np.round(ref[n], 6)) for n in sizes], "first n indices of each training fold; average over folds"),
        ("training error rises and validation error falls with more data", lambda: (
            bool(run()["train_rmse"].iloc[0] < run()["train_rmse"].iloc[-1]), bool(run()["val_rmse"].iloc[0] > run()["val_rmse"].iloc[-1])),
         (True, True), None),
        ("the model passed in is not fitted", model_unfitted, False, "fit clone(model), not the model itself"),
    ])


def check_bias_variance(f):
    x_eval = np.linspace(0, 4, 40)

    def truth(x):
        return np.sin(1.5 * x)

    def flat(x):
        return np.full_like(x, 2.0)

    def run(fit_predict, tr=truth, n_train=20, repeats=200, seed=0):
        return f(fit_predict, tr, x_eval, 0.5, n_train, repeats, np.random.default_rng(seed))

    def perfect():
        return tuple(round(v, 12) for v in run(lambda xt, yt, xe: truth(xe)))

    def zero():
        b, v = run(lambda xt, yt, xe: np.zeros_like(xe))
        return (round(b, 10), round(v, 10))

    def mean_model_variance():
        b, v = run(lambda xt, yt, xe: np.full_like(xe, yt.mean()), tr=flat, n_train=25, repeats=4000)
        return 0.8 < v / (0.5 ** 2 / 25) < 1.2 and b < 0.002

    def reproducible():
        fp = lambda xt, yt, xe: np.polynomial.Polynomial.fit(xt, yt, 3)(xe)
        return run(fp, seed=3) == run(fp, seed=3)

    def calls():
        count = []

        def fp(xt, yt, xe):
            count.append((len(xt), len(yt)))
            return np.zeros_like(xe)
        run(fp, n_train=15, repeats=7)
        return count

    _scenarios("bias_variance", [
        ("a model that returns the truth: no bias, no variance", perfect, (0.0, 0.0), None),
        ("always predicting 0: bias² = mean(truth²), no variance", zero, (round(float(np.mean(truth(x_eval) ** 2)), 10), 0.0),
         "bias² compares the average prediction with truth(x_eval)"),
        ("predicting mean(y_train): variance close to noise² / n", mean_model_variance, True,
         "variance across repeats for each point (axis=0), then averaged over the points"),
        ("calls fit_predict n_repeats times with n_train points", calls, [(15, 15)] * 7, None),
        ("same seed, same result", reproducible, True, "use the rng passed in for all the randomness"),
    ])


def check_vif(f):
    X, _ = load_diabetes(return_X_y=True, as_frame=True)

    def independent():
        rng = np.random.default_rng(0)
        data = pd.DataFrame(rng.normal(size=(2000, 3)), columns=["a", "b", "c"])
        return bool(np.all(f(data).between(1.0, 1.02)))

    def known():
        rng = np.random.default_rng(1)
        a = rng.normal(size=5000)
        b = a + rng.normal(size=5000)               # R² of a on b is about 0.5, so VIF about 2
        return bool(np.all(f(pd.DataFrame({"a": a, "b": b})).between(1.85, 2.15)))

    def duplicate():
        data = pd.DataFrame({"a": [1.0, 2.0, 3.0, 5.0], "b": [2.0, 4.0, 6.0, 10.0], "c": [1.0, 0.0, 1.0, 3.0]})
        return bool(np.isinf(f(data)["a"]) and np.isinf(f(data)["b"]))

    _scenarios("vif", [
        ("unrelated features have VIF close to 1", independent, True, None),
        ("two related features: VIF close to the theoretical 2", known, True, "VIF = 1 / (1 - R²), regressing each column on the others"),
        ("diabetes: s1 and s2 are strongly inflated", lambda: bool(f(X)["s1"] > 20 and f(X)["s2"] > 20), True, None),
        ("indexed by column name, in order", lambda: list(f(X).index), list(X.columns), None),
        ("a perfectly redundant feature gets inf", duplicate, True, "return np.inf when R² is 1"),
    ])


def check_lasso_coordinate_descent(f):
    X, y = load_diabetes(return_X_y=True)
    Xs = StandardScaler().fit_transform(X)

    def matches(alpha):
        w, b = f(Xs, y, alpha)
        ref = Lasso(alpha=alpha, tol=1e-12, max_iter=100_000).fit(Xs, y)
        same_zeros = bool(np.array_equal(w == 0, ref.coef_ == 0))
        return (bool(np.allclose(w, ref.coef_, atol=1e-4)), abs(b - ref.intercept_) < 1e-4, same_zeros)

    def huge():
        w, b = f(Xs, y, 1000.0)
        return (bool(np.all(w == 0)), round(b, 6) == round(float(y.mean()), 6))

    def small_is_ols():
        w, _ = f(Xs, y, 1e-6)
        return bool(np.allclose(w, LinearRegression().fit(Xs, y).coef_, atol=1e-2))

    _scenarios("lasso_coordinate_descent", [
        ("matches scikit-learn's Lasso, alpha = 1 (weights, intercept, zeros)", lambda: matches(1.0), (True, True, True),
         "w_j = S(rho_j, alpha) / z_j with rho_j computed from the current residual"),
        ("matches at alpha = 10, with more weights at exactly zero", lambda: matches(10.0), (True, True, True),
         "S(rho, alpha) = sign(rho) * max(|rho| - alpha, 0) gives exact zeros"),
        ("a huge alpha removes every feature; intercept is mean(y)", huge, (True, True), None),
        ("a tiny alpha is close to least squares", small_is_ols, True, None),
    ])


def check_softmax_regression(f):
    rng = np.random.default_rng(4)
    centres = np.array([[0.0, 0.0], [2.5, 0.5], [1.0, 2.5]])
    y = np.repeat([0, 1, 2], 60)
    X = centres[y] + rng.normal(0, 0.8, size=(180, 2))
    C = 1.0

    out = {}

    def run():
        if "v" not in out:
            out["v"] = f(X, y, 3, lr=0.5, epochs=3000, l2=1 / (C * len(y)))
        return out["v"]

    def proba(W, b):
        Z = X @ W + b
        Z = Z - Z.max(axis=1, keepdims=True)
        P = np.exp(Z)
        return P / P.sum(axis=1, keepdims=True)

    def matches_sklearn():
        W, b, _ = run()
        ref = LogisticRegression(C=C, tol=1e-10, max_iter=10_000).fit(X, y)
        return bool(np.abs(proba(W, b) - ref.predict_proba(X)).max() < 2e-3)

    _scenarios("softmax_regression", [
        ("shapes: W (d, k), b (k,), one loss per epoch", lambda: (run()[0].shape, run()[1].shape, len(run()[2])), ((2, 3), (3,), 3000), None),
        ("the first loss is log(3), the uniform guess", lambda: round(float(run()[2][0]), 6), round(math.log(3), 6),
         "record the loss before the first update; start from zeros"),
        ("the loss goes down", lambda: bool(run()[2][-1] < run()[2][0] * 0.5 and np.all(np.diff(run()[2]) <= 1e-12)), True,
         "dZ = (P - Y) / n, dW = Xᵀ dZ + l2·W, db = dZ.sum(axis=0)"),
        ("matches scikit-learn's LogisticRegression with l2 = 1 / (C·n)", matches_sklearn, True,
         "penalise W only, with (l2 / 2)·ΣW² in the loss and l2·W in the gradient"),
        ("classifies the training data well", lambda: float(np.mean(np.argmax(proba(*run()[:2]), axis=1) == y)) > 0.85, True, None),
    ])


def check_target_encode_oof(f):
    cats = np.array(["a", "a", "a", "b", "b", "c", "c", "c", "c", "d"])
    y = np.array([1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0])

    def reference(categories, target, n_splits, smoothing, seed):
        enc = np.empty(len(target))
        for fi, ei in KFold(n_splits, shuffle=True, random_state=seed).split(categories):
            m = target[fi].mean()
            df = pd.DataFrame({"c": categories[fi], "y": target[fi]}).groupby("c")["y"].agg(["sum", "count"])
            mp = ((df["sum"] + smoothing * m) / (df["count"] + smoothing)).to_dict()
            enc[ei] = [mp.get(c, m) for c in categories[ei]]
        return enc

    def own_target_excluded():
        enc, _, _ = f(cats, y, n_splits=5, smoothing=0.0, seed=0)
        lone = int(np.where(cats == "d")[0][0])
        others = np.ones(len(y), bool)
        for fi, ei in KFold(5, shuffle=True, random_state=0).split(cats):
            if lone in ei:
                return round(enc[lone], 10) == round(float(y[fi].mean()), 10)
        return False

    def leakage():
        rng = np.random.default_rng(0)
        c = rng.integers(0, 1000, 2000).astype(str)          # about two rows per category
        target = rng.normal(size=2000)                     # pure noise: nothing to learn
        enc, _, _ = f(c, target, n_splits=5, smoothing=0.0, seed=1)
        naive = pd.Series(target).groupby(c).transform("mean").to_numpy()
        return (bool(abs(np.corrcoef(enc, target)[0, 1]) < 0.1), bool(np.corrcoef(naive, target)[0, 1] > 0.5))

    def full_mapping():
        _, mapping, m = f(cats, y, n_splits=5, smoothing=2.0, seed=0)
        return ({k: round(v, 6) for k, v in mapping.items()}, round(m, 6))

    gm = y.mean()
    expected_map = {"a": round((2 + 2 * gm) / (3 + 2), 6), "b": round((0 + 2 * gm) / (2 + 2), 6),
                    "c": round((3 + 2 * gm) / (4 + 2), 6), "d": round((1 + 2 * gm) / (1 + 2), 6)}
    _scenarios("target_encode_oof", [
        ("matches a reference, smoothing 2", lambda: np.round(f(cats, y, n_splits=5, smoothing=2.0, seed=0)[0], 10),
         np.round(reference(cats, y, 5, 2.0, 0), 10), "for each fold, compute the smoothed means from the other folds only"),
        ("a category seen once gets the other folds' mean, not its own target", own_target_excluded, True,
         "never include a row's own target in its encoding"),
        ("the mapping for new data uses all rows", full_mapping, (expected_map, round(gm, 6)), None),
        ("on pure noise: out-of-fold shows no signal, in-sample would fake one", leakage, (True, True), None),
        ("lists work too", lambda: len(f(list(cats), list(y))[0]), 10, None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `regression_metrics`

In [2]:
def regression_metrics(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.shape != y_pred.shape or y_true.size == 0:
        raise ValueError("y_true and y_pred must be non-empty and the same length")
    errors = y_true - y_pred
    total = np.sum((y_true - y_true.mean()) ** 2)
    if total == 0:
        raise ValueError("R² is undefined when every true value is the same")
    return {
        "mae": float(np.mean(np.abs(errors))),
        "rmse": float(np.sqrt(np.mean(errors ** 2))),
        "r2": float(1 - np.sum(errors ** 2) / total),
    }

In [3]:
check_regression_metrics(regression_metrics)

PASS  all 8 tests passed for regression_metrics


Writing the formulas yourself makes their behaviour obvious:

- R² compares the model with **predicting the mean of `y_true`**. A model that's worse than that gets a negative R², which is why the check includes a case with R² of -3. On held-out data this happens more often than people expect.
- RMSE is always at least as large as MAE, and equal only when every error has the same size.

One subtlety worth knowing: R² on a **test set** uses the test set's own mean as the baseline, a value the model never saw. scikit-learn's `r2_score` does the same. It's the standard definition, but it slightly flatters the baseline, so R² on a small test set should be read with care.

### Exercise 2: `polynomial_features`

In [4]:
def polynomial_features(x, degree: int):
    if degree < 1:
        raise ValueError(f"degree must be at least 1, got {degree}")
    x = np.asarray(x, dtype=float)
    return x[:, None] ** np.arange(1, degree + 1)

In [5]:
check_polynomial_features(polynomial_features)

PASS  all 6 tests passed for polynomial_features


`x[:, None]` turns the length-`n` vector into an `(n, 1)` column, and `np.arange(1, degree + 1)` is a row of powers. Broadcasting (Module 6) raises every value to every power in one step, giving the `(n, degree)` table.

Leaving out the column of ones (`x⁰`) is deliberate: the model's intercept already plays that role. scikit-learn's `PolynomialFeatures` includes it by default (`include_bias=True`), which is harmless with `LinearRegression` but worth knowing about.

High powers grow fast: with `x = 10` and degree 9, the last column is a billion, while the first is 10. That's why polynomial features are almost always followed by a `StandardScaler`, and why splines are often the better choice for flexible curves.

### Exercise 3: `fit_least_squares`

In [6]:
def fit_least_squares(X, y):
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X[:, None]
    A = np.column_stack([X, np.ones(len(X))])
    coefficients = np.linalg.lstsq(A, np.asarray(y, dtype=float), rcond=None)[0]
    return coefficients[:-1], float(coefficients[-1])

In [7]:
check_fit_least_squares(fit_least_squares)

PASS  all 5 tests passed for fit_least_squares


Why `lstsq` and not the normal equations, `np.linalg.solve(A.T @ A, A.T @ y)`? With a duplicated column, `AᵀA` is **singular** (it has no inverse), so `solve` raises an error or returns numerical garbage. `lstsq` handles it: when many weight vectors fit equally well, it returns the one with the smallest size, which here splits the weight evenly between the two copies. The predictions are correct either way.

Exact duplicates are rare, but near-duplicates (the lesson's `s1` and `s2`) are common, and they cause the same trouble in a milder form: the normal equations amplify rounding errors. `lstsq` uses a more careful method (based on the singular value decomposition) that avoids forming `AᵀA` at all.

### Exercise 4: `add_interactions`

In [8]:
def add_interactions(df: pd.DataFrame, pairs) -> pd.DataFrame:
    result = df.copy()
    for a, b in pairs:
        missing = [c for c in (a, b) if c not in df.columns]
        if missing:
            raise KeyError(f"columns not found: {missing}")
        result[f"{a}*{b}"] = df[a] * df[b]
    return result

In [9]:
check_add_interactions(add_interactions)

PASS  all 5 tests passed for add_interactions


An interaction lets the effect of one feature depend on another. With `income` and `rooms` alone, a linear model says an extra room adds the same amount everywhere. With `income*rooms` as well, the value of an extra room can be larger in richer areas.

Interactions multiply quickly: 20 features have 190 pairs. Adding them all is what `PolynomialFeatures(interaction_only=True)` does, and it's exactly the situation where regularisation becomes essential, as the lesson's 65-feature example showed. Choosing a few interactions that make sense for the problem is usually better than adding all of them.

`df.copy()` matters: assigning a new column to `df` itself would change the caller's DataFrame, a side effect that's easy to miss in a notebook until a later cell suddenly sees extra columns.

## Medium

### Exercise 5: `ridge_closed_form`

In [10]:
def ridge_closed_form(X, y, alpha: float):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    X_mean, y_mean = X.mean(axis=0), y.mean()
    Xc, yc = X - X_mean, y - y_mean
    w = np.linalg.solve(Xc.T @ Xc + alpha * np.eye(X.shape[1]), Xc.T @ yc)
    return w, float(y_mean - X_mean @ w)

In [11]:
check_ridge_closed_form(ridge_closed_form)

PASS  all 5 tests passed for ridge_closed_form


**Why centring removes the intercept from the problem.** Once every column of `X` and `y` has mean zero, the best intercept for the centred data is exactly 0, whatever `w` is. So `w` can be solved for alone, with the penalty applied only to it, and the intercept recovered afterwards from the means.

**What goes wrong if the intercept is penalised.** The check uses a huge `alpha`: the weights should shrink to about zero and the prediction become the mean of `y`. If the intercept were penalised too, it would also be pushed to zero, and the model would predict 0 for everything, which is very wrong when `y` averages 150.

**Why `alpha · I` fixes singular matrices.** Adding a positive number to every diagonal entry makes the matrix invertible, however correlated or duplicated the columns are. The check includes a duplicated column with `alpha = 1`, which this method handles, unlike the plain normal equations.

### Exercise 6: `choose_alpha`

In [12]:
def choose_alpha(X, y, alphas, cv):
    rmse = {}
    for alpha in alphas:
        model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
        scores = cross_val_score(model, X, y, cv=cv, scoring="neg_root_mean_squared_error")
        rmse[alpha] = -scores.mean()
    rmse = pd.Series(rmse)
    return rmse.idxmin(), rmse

In [13]:
check_choose_alpha(choose_alpha)

PASS  all 3 tests passed for choose_alpha


The check compares your numbers with a reference to six decimal places, so it can tell whether the scaler is inside the pipeline. Scaling the whole of `X` first and then cross-validating gives **slightly different** numbers, because each fold's scaler would have seen the validation rows' means and spreads. Here the difference is tiny; with steps that learn more from the data (Module 11's feature selection), it isn't.

scikit-learn's convention of making every score "higher is better" is why the RMSE comes back negative (`neg_root_mean_squared_error`). It's slightly awkward, but it means the same code can maximise any score.

`RidgeCV` does this search much faster for ridge specifically, using a mathematical shortcut that avoids refitting for every fold. Writing it with `cross_val_score` works for **any** model and any hyperparameter.

### Exercise 7: `learning_curve_points`

In [14]:
from sklearn.base import clone


def learning_curve_points(model, X, y, sizes, cv) -> pd.DataFrame:
    rows = {}
    splits = list(cv.split(X))
    for n in sizes:
        train_errors, val_errors = [], []
        for train_idx, val_idx in splits:
            subset = train_idx[:n]
            fitted = clone(model).fit(X[subset], y[subset])
            train_errors.append(np.sqrt(np.mean((fitted.predict(X[subset]) - y[subset]) ** 2)))
            val_errors.append(np.sqrt(np.mean((fitted.predict(X[val_idx]) - y[val_idx]) ** 2)))
        rows[n] = {"train_rmse": np.mean(train_errors), "val_rmse": np.mean(val_errors)}
    return pd.DataFrame.from_dict(rows, orient="index")

In [15]:
check_learning_curve_points(learning_curve_points)

PASS  all 4 tests passed for learning_curve_points


`clone(model)` makes an unfitted copy with the same settings. Without it, every fit would reuse the same object. With most scikit-learn models that only wastes the previous fit, but with models that can continue training from where they stopped (`warm_start=True`), reusing one object would silently carry learning from one size to the next.

The validation set stays the **same** for every size within a split; only the training subset grows. That's what makes the curve readable: changes come from the amount of training data, not from a changing test.

The check runs it with a straight line on curved data (high bias: the two errors meet quickly and stay high) and confirms the training error rises while the validation error falls as `n` grows, the shapes from lesson section 7. `sklearn.model_selection.learning_curve` does the same job with more options, such as running the fits in parallel.

### Exercise 8: `bias_variance`

In [16]:
def bias_variance(fit_predict, truth, x_eval, noise_sd: float, n_train: int, n_repeats: int, rng):
    x_eval = np.asarray(x_eval, dtype=float)
    low, high = x_eval.min(), x_eval.max()
    predictions = []
    for _ in range(n_repeats):
        x_train = rng.uniform(low, high, n_train)
        y_train = truth(x_train) + rng.normal(0, noise_sd, n_train)
        predictions.append(fit_predict(x_train, y_train, x_eval))
    predictions = np.array(predictions)
    bias_sq = np.mean((predictions.mean(axis=0) - truth(x_eval)) ** 2)
    variance = np.mean(predictions.var(axis=0))
    return float(bias_sq), float(variance)

In [17]:
check_bias_variance(bias_variance)

PASS  all 5 tests passed for bias_variance


The checks use models whose answers are known exactly, so they don't depend on matching anyone's random numbers:

- A "model" that returns the true curve has bias 0 and variance 0.
- One that always predicts 0 has variance 0 and bias² equal to the average of `truth²`.
- One that predicts the **mean of `y_train`** for a flat truth has bias 0 and variance `noise_sd² / n_train`, the familiar variance of an average (Module 9's standard error, squared).

That last case shows the core trade-off in miniature: averaging more points reduces variance in proportion to `1 / n`. Every way of fighting variance, more data, regularisation, averaging many models (Module 14's random forests), is a version of that idea.

Getting the axis right matters: `predictions.mean(axis=0)` averages **across repeats** for each evaluation point. Averaging along the other axis would compute something meaningless, and the known-answer checks catch it.

### Exercise 9: `vif`

In [18]:
def vif(X: pd.DataFrame) -> pd.Series:
    values = {}
    for column in X.columns:
        others = X.drop(columns=column)
        target = X[column]
        r2 = LinearRegression().fit(others, target).score(others, target)
        values[column] = np.inf if r2 >= 1 - 1e-12 else 1 / (1 - r2)
    return pd.Series(values)

In [19]:
check_vif(vif)

PASS  all 5 tests passed for vif


The name comes from what it measures: how many times larger the **variance of that feature's weight** is than it would be if the feature were uncorrelated with the rest. A VIF of 9 means the weight's standard error is three times bigger (the square root of 9), so its value is far less certain.

On the diabetes data, the check expects `s1` and `s2` to have VIFs in the dozens, and `s3`, `s4` and `s5` to be high too, since several of the serum measurements are related. Those are exactly the features whose least squares weights behaved strangely.

What to do about high VIFs depends on the goal. If you only need **predictions**, often nothing: multicollinearity hurts interpretation more than accuracy. If you need **interpretable weights**, use ridge, drop or combine redundant features, or report the group's combined effect.

The `1e-12` tolerance treats R² values that are 1 up to rounding as exactly 1, so perfectly redundant columns get `inf` rather than an enormous finite number that depends on rounding errors.

## Hard

### Exercise 10: `lasso_coordinate_descent`

In [20]:
def soft_threshold(rho, alpha):
    return np.sign(rho) * max(abs(rho) - alpha, 0.0)


def lasso_coordinate_descent(X, y, alpha: float, n_iter: int = 1000, tol: float = 1e-10):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, d = X.shape
    X_mean, y_mean = X.mean(axis=0), y.mean()
    Xc, yc = X - X_mean, y - y_mean
    z = (Xc ** 2).sum(axis=0) / n
    w = np.zeros(d)
    residual = yc.copy()                       # yc - Xc @ w, with w = 0
    for _ in range(n_iter):
        biggest_change = 0.0
        for j in range(d):
            if z[j] == 0:
                continue                       # a constant column can't help
            rho = Xc[:, j] @ (residual + Xc[:, j] * w[j]) / n
            new_w = soft_threshold(rho, alpha) / z[j]
            change = new_w - w[j]
            if change != 0.0:
                residual -= Xc[:, j] * change  # keep the residual in step with w
                w[j] = new_w
                biggest_change = max(biggest_change, abs(change))
        if biggest_change < tol:
            break
    return w, float(y_mean - X_mean @ w)

In [21]:
check_lasso_coordinate_descent(lasso_coordinate_descent)

PASS  all 4 tests passed for lasso_coordinate_descent


**Where the zeros come from.** Soft-thresholding is the heart of the lasso. `ρ_j` measures how much feature `j` would like to explain of what's left of `y`. If `|ρ_j|` is smaller than `α`, the feature isn't worth its penalty, and its weight is set to **exactly** zero; otherwise it's shrunk towards zero by `α`. Ridge's squared penalty never produces an exact zero, because its pull weakens as the weight gets small; the absolute value pulls just as hard however small the weight is.

**Keeping the residual up to date** is what makes coordinate descent fast. Recomputing `yc - Xc @ w` after every single update would cost `n × d` operations each time; adjusting it by the one column that changed costs only `n`.

**Why it converges.** Each update makes the objective smaller or leaves it the same, and the lasso objective is convex (one valley, no false minima), so the sweeps settle at the true minimum. scikit-learn's `Lasso` uses the same algorithm, written in fast compiled code, and the check compares your weights with it, including which ones are exactly zero.

### Exercise 11: `softmax_regression`

In [22]:
def softmax_regression(X, y, n_classes: int, lr: float = 0.5, epochs: int = 3000, l2: float = 0.0):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    n, d = X.shape
    Y = np.eye(n_classes)[y]                         # one-hot labels, shape (n, k)
    W = np.zeros((d, n_classes))
    b = np.zeros(n_classes)
    losses = []
    for _ in range(epochs):
        Z = X @ W + b
        Z = Z - Z.max(axis=1, keepdims=True)         # stable softmax
        P = np.exp(Z)
        P /= P.sum(axis=1, keepdims=True)
        loss = -np.mean(np.log(P[np.arange(n), y] + 1e-15)) + 0.5 * l2 * np.sum(W ** 2)
        losses.append(loss)
        dZ = (P - Y) / n
        W -= lr * (X.T @ dZ + l2 * W)
        b -= lr * dZ.sum(axis=0)
    return W, b, np.array(losses)

In [23]:
check_softmax_regression(softmax_regression)

PASS  all 5 tests passed for softmax_regression


**The gradient is the same tidy result as for two classes.** For softmax with cross-entropy, the derivative of the loss with respect to the scores is just `P - Y`: predicted probabilities minus the one-hot truth. Module 9 derived this for the binary case (`p - y`); the multi-class version is the same idea, one column per class. Everything else is the chain rule through `Z = XW + b`.

**`np.eye(k)[y]`** builds the one-hot matrix in one line: row `i` of the identity matrix is the one-hot vector for class `i`, and indexing with the label array picks the right row for every example.

**Matching scikit-learn.** scikit-learn writes the penalised objective as `C · (sum of losses) + ½‖W‖²`. Dividing by `C · n` gives the mean loss plus `‖W‖² / (2Cn)`, so `l2 = 1 / (C · n)` is the same objective, and a converged solution must be the same. Without any penalty, softmax weights aren't unique (adding the same vector to every class's weights changes nothing), but the L2 penalty picks one solution, so the comparison is well defined.

**Initialising at zero** is fine here, unlike in neural networks: the loss is convex, so there's one minimum, and the first gradient step already breaks the symmetry between classes. The first recorded loss is `log(k)`, the uniform guess, which the check uses to confirm the loss is recorded before the update.

### Exercise 12: `target_encode_oof`

In [24]:
from sklearn.model_selection import KFold


def _smoothed_means(categories, y, smoothing):
    m = y.mean()
    table = pd.DataFrame({"category": categories, "y": y}).groupby("category")["y"].agg(["sum", "count"])
    values = (table["sum"] + smoothing * m) / (table["count"] + smoothing)
    return values.to_dict(), float(m)


def target_encode_oof(categories, y, n_splits: int = 5, smoothing: float = 10.0, seed: int = 0):
    categories = np.asarray(categories)
    y = np.asarray(y, dtype=float)
    encoded = np.empty(len(y))
    for fit_idx, encode_idx in KFold(n_splits, shuffle=True, random_state=seed).split(categories):
        mapping, m = _smoothed_means(categories[fit_idx], y[fit_idx], smoothing)
        encoded[encode_idx] = [mapping.get(c, m) for c in categories[encode_idx]]
    mapping, global_mean = _smoothed_means(categories, y, smoothing)
    return encoded, mapping, global_mean

In [25]:
check_target_encode_oof(target_encode_oof)

PASS  all 5 tests passed for target_encode_oof


**Why out of fold.** Imagine a category that appears once. Its in-sample average **is** that row's target, so the encoded feature equals the answer exactly, and the model will happily learn to copy it. Every rare category leaks a little the same way. The check builds a dataset where the target is pure noise and the categories are random: a naive in-sample encoding still "predicts" the noise well (strong correlation), while the out-of-fold encoding shows the truth, correlation near zero.

**Why smoothing.** A category seen twice has a very unreliable average. Smoothing pulls it towards the overall mean as if `smoothing` extra rows with the average target had been added. Categories with lots of data keep their own mean; rare ones mostly get the global one.

**Training and prediction use different encodings, on purpose.** The training rows get out-of-fold values, so the model learns how much to trust an encoding that wasn't computed from the row itself. New data gets the encoding from all the training data, which is the best estimate available. Libraries such as scikit-learn's `TargetEncoder` follow exactly this scheme, using the same cross-fitting idea.

## What's next

The mini project in `04_mini_project.ipynb` applies everything here to the 1990 California census: a ladder of linear models, feature engineering, ridge regularisation, learning curves, and an honest look at how geography affects evaluation.